# OOF (Out-of-Fold) 스태킹 기반 수요예측 보정 모델

## 방법론 개요

**OOF Stacking**은 기존 Residual 보정 방식과 달리, K-Fold 교차검증을 통해 생성된  
Out-of-Fold 예측값을 메타 피처로 활용하는 2단계 앙상블 방법론이다.

### Residual 방식 vs OOF 방식 비교

| 구분 | Residual 방식 | OOF 방식 |
|------|-------------|----------|
| 베이스 예측 | 전체 학습셋으로 학습 | K-Fold 별 학습 |
| 메타 입력 | Residual (잔차) | OOF 예측값 (원본 스케일) |
| 과적합 위험 | 높음 (In-sample 잔차 사용) | 낮음 (Out-of-sample 예측 사용) |
| 메타 모델 | Residual 보정 모델 | Ridge 회귀 (최적 가중치 학습) |

### 구현 단계
1. **Base Models**: Naive Drift, GradientBoosting, LightGBM
2. **OOF 생성**: `TimeSeriesSplit`으로 학습셋 내 K-Fold 예측 생성
3. **Meta Model**: Ridge Regression으로 최적 앙상블 가중치 학습 (OOF 예측이 메타 피처)
4. **최종 예측**: Base 모델 전체 데이터로 재학습 → 메타 모델 적용

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (14, 5)
print('라이브러리 로드 완료')

In [ ]:
# ── 설정값 ──────────────────────────────────────────
DATA_PATH     = 'data_weekly_260120.csv'
TARGET_COL    = 'Com_LME_Ni_Cash'

VAL_START     = pd.to_datetime('2025-08-04')
VAL_END       = pd.to_datetime('2025-10-20')
TEST_START    = pd.to_datetime('2025-10-27')
TEST_END      = pd.to_datetime('2026-01-12')

N_SPLITS      = 5          # OOF TimeSeriesSplit 폴드 수
RANDOM_STATE  = 42
BASELINE_RMSE = 406.80     # sparta2_advanced.ipynb 기준 (Naive 0.8 + GB 0.2)

print(f'설정 완료 | N_SPLITS={N_SPLITS} | Baseline RMSE={BASELINE_RMSE}')

## 1. 데이터 로딩

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=['dt'])
df_raw = df_raw.set_index('dt').sort_index()

print(f'데이터 기간  : {df_raw.index[0].date()} ~ {df_raw.index[-1].date()}')
print(f'전체 행/열   : {df_raw.shape}')
print(f'결측치 수    : {df_raw.isnull().sum().sum()}')
print(f'타겟 컬럼    : {TARGET_COL}')

## 2. 피처 엔지니어링

`sparta2_advanced.ipynb`와 동일한 방식으로 피처를 구성한다.  
- **1주 shift**: 미래 정보 유출(Leakage) 방지  
- **추가 피처**: 실현변동성(RV), 변화율(ROC), Z-Score, 래그 수익률

In [ ]:
feature_cols = [c for c in df_raw.columns if c != TARGET_COL]

# 1주 shift로 leakage 방지
X_raw = df_raw[feature_cols].shift(1)
y_raw = df_raw[TARGET_COL]

X_eng = X_raw.copy()

# ① 실현변동성 (Realized Volatility)
log_ret = np.log(y_raw / y_raw.shift(1))
for w in [4, 8, 12, 26]:
    X_eng[f'RV_{w}w'] = log_ret.shift(1).rolling(w).std() * np.sqrt(52)

# ② 변화율 (Rate of Change)
for w in [4, 12, 26]:
    X_eng[f'ROC_{w}w'] = y_raw.shift(1).pct_change(w)

# ③ Z-Score
for w in [4, 26]:
    roll_mean = y_raw.shift(1).rolling(w).mean()
    roll_std  = y_raw.shift(1).rolling(w).std()
    X_eng[f'zscore_{w}w'] = (y_raw.shift(1) - roll_mean) / (roll_std + 1e-8)

# ④ 래그 수익률
for lag in [1, 2, 3]:
    X_eng[f'ret_lag_{lag}'] = log_ret.shift(lag)

# 결측치 제거
valid = X_eng.notna().all(axis=1) & y_raw.notna()
X = X_eng[valid]
y = y_raw[valid]

print(f'유효 행 수  : {len(X)}')
print(f'피처 수     : {X.shape[1]} (원본 {len(feature_cols)} + 추가 12)')

## 3. Train / Validation / Test 분할

In [ ]:
train_mask = X.index < VAL_START
val_mask   = (X.index >= VAL_START) & (X.index <= VAL_END)
test_mask  = (X.index >= TEST_START) & (X.index <= TEST_END)

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train : {len(X_train):4d}개  ({X_train.index[0].date()} ~ {X_train.index[-1].date()})')
print(f'Val   : {len(X_val):4d}개  ({X_val.index[0].date()} ~ {X_val.index[-1].date()})')
print(f'Test  : {len(X_test):4d}개  ({X_test.index[0].date()} ~ {X_test.index[-1].date()})')

## 4. 베이스라인 모델 (비교 기준)

OOF 스태킹과 비교할 기준 모델들을 먼저 학습한다.

In [ ]:
def naive_drift_multi(y_series, n_steps, damping=1.0):
    """1-Step Naive Drift를 n_steps번 반복 예측"""
    last  = y_series.iloc[-1]
    drift = (y_series.iloc[-1] - y_series.iloc[-2]) * damping
    return np.array([last + (i + 1) * drift for i in range(n_steps)])


def eval_metrics(y_true, y_pred, name=''):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    print(f'  [{name:38s}]  RMSE={rmse:7.2f}  MAE={mae:7.2f}  MAPE={mape:.2f}%')
    return dict(model=name, rmse=rmse, mae=mae, mape=mape)


results = []   # 최종 성능 수집용

# ── Naive Drift ──────────────────────────────────────
naive_val  = naive_drift_multi(y_train, len(y_val))
naive_test = naive_drift_multi(y_val,   len(y_test))

# ── GradientBoosting (전체 학습셋) ───────────────────
gb_base = GradientBoostingRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE
)
gb_base.fit(X_train.values, y_train.values)
gb_val  = gb_base.predict(X_val.values)
gb_test = gb_base.predict(X_test.values)

# ── LightGBM (전체 학습셋) ───────────────────────────
lgb_base = lgb.LGBMRegressor(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    random_state=RANDOM_STATE, verbose=-1
)
lgb_base.fit(X_train.values, y_train.values)
lgb_val  = lgb_base.predict(X_val.values)
lgb_test = lgb_base.predict(X_test.values)

# ── Baseline Hybrid (0.8 Naive + 0.2 GB) ────────────
hybrid_val  = 0.8 * naive_val  + 0.2 * gb_val
hybrid_test = 0.8 * naive_test + 0.2 * gb_test

print('=== Test Set 베이스라인 성능 ===')
results.append(eval_metrics(y_test, naive_test,  'Naive Drift'))
results.append(eval_metrics(y_test, gb_test,     'GradientBoosting'))
results.append(eval_metrics(y_test, lgb_test,    'LightGBM'))
results.append(eval_metrics(y_test, hybrid_test, 'Hybrid Baseline (0.8N + 0.2GB)'))
print(f'\n  [참고] sparta2_advanced 기준선: RMSE = {BASELINE_RMSE:.2f}')

## 5. OOF (Out-of-Fold) 예측 생성

### 핵심 아이디어
학습 데이터를 `TimeSeriesSplit`으로 K개 폴드로 나눠,  
각 폴드에서 **홀드아웃(Val) 구간에 대한 예측**을 수집한다.  
이 OOF 예측은 메타 모델의 학습 데이터로 사용된다.

```
Fold 1:  [====Train====] [--Val--] [ ... ]
Fold 2:  [=====Train=====] [--Val--] [ ... ]
Fold 3:  [======Train======] [--Val--] [ ... ]
               ↓ 각 Val 구간 예측 수집
         [   OOF Predictions (전체 학습셋 커버)   ]
               ↓ 메타 모델 학습
         Meta Model: Ridge(OOF_Naive, OOF_GB, OOF_LGB) → y
```

### Naive Drift OOF 처리
- 각 폴드의 학습셋 마지막 두 값으로 drift 계산
- Val 구간 전체에 다단계 외삽 적용

In [ ]:
tscv      = TimeSeriesSplit(n_splits=N_SPLITS)
X_tr_arr  = X_train.values
y_tr_arr  = y_train.values

oof_naive = np.zeros(len(X_train))
oof_gb    = np.zeros(len(X_train))
oof_lgb   = np.zeros(len(X_train))
oof_mask  = np.zeros(len(X_train), dtype=bool)   # OOF 예측이 채워진 인덱스

print(f'TimeSeriesSplit(n_splits={N_SPLITS}) OOF 생성 시작\n')

for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_tr_arr)):
    X_fold_tr  = X_tr_arr[tr_idx]
    X_fold_val = X_tr_arr[val_idx]
    y_fold_tr  = y_tr_arr[tr_idx]

    print(f'  Fold {fold_idx+1}/{N_SPLITS}  |  Train={len(tr_idx):4d}개  '
          f'(~{X_train.index[tr_idx[-1]].date()})  '
          f'Val={len(val_idx):4d}개  '
          f'({X_train.index[val_idx[0]].date()} ~ {X_train.index[val_idx[-1]].date()})')

    # ── Naive Drift OOF (다단계 외삽) ──────────────
    last_p  = y_tr_arr[tr_idx[-1]]
    drift_p = last_p - y_tr_arr[tr_idx[-2]]
    oof_naive[val_idx] = [last_p + (i + 1) * drift_p for i in range(len(val_idx))]

    # ── GradientBoosting OOF ───────────────────────
    gb_fold = GradientBoostingRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5, random_state=RANDOM_STATE
    )
    gb_fold.fit(X_fold_tr, y_fold_tr)
    oof_gb[val_idx] = gb_fold.predict(X_fold_val)

    # ── LightGBM OOF ───────────────────────────────
    lgb_fold = lgb.LGBMRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        random_state=RANDOM_STATE, verbose=-1
    )
    lgb_fold.fit(X_fold_tr, y_fold_tr)
    oof_lgb[val_idx] = lgb_fold.predict(X_fold_val)

    oof_mask[val_idx] = True

print(f'\nOOF 생성 완료  |  유효 샘플: {oof_mask.sum()}개 / {len(X_train)}개')

## 6. OOF 내부 성능 확인

OOF 예측이 Out-of-Sample에서 얼마나 잘 동작하는지 확인한다.  
이 성능은 메타 모델이 학습하는 실제 품질을 반영한다.

In [ ]:
y_oof_true = y_tr_arr[oof_mask]

print('=== OOF 내부 성능 (학습셋 내 교차검증) ===')
eval_metrics(y_oof_true, oof_naive[oof_mask], 'OOF Naive Drift')
eval_metrics(y_oof_true, oof_gb[oof_mask],    'OOF GradientBoosting')
eval_metrics(y_oof_true, oof_lgb[oof_mask],   'OOF LightGBM')

# OOF 예측 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
oof_labels = ['Naive Drift', 'GradientBoosting', 'LightGBM']
oof_preds  = [oof_naive[oof_mask], oof_gb[oof_mask], oof_lgb[oof_mask]]

for ax, pred, label in zip(axes, oof_preds, oof_labels):
    ax.plot(y_oof_true, label='실제값', color='black', alpha=0.7, linewidth=1)
    ax.plot(pred,       label='OOF 예측', alpha=0.8)
    ax.set_title(f'OOF — {label}', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel('OOF 샘플 인덱스')
    ax.set_ylabel('Nickel Price ($/ton)')
    ax.grid(alpha=0.3)

plt.suptitle('OOF 예측 vs 실제값 (학습셋 내 K-Fold 결과)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. 메타 모델 학습 (Ridge Regression)

OOF 예측값 3개를 피처로, 실제 니켈 가격을 타겟으로 Ridge 회귀를 학습한다.  
Ridge는 계수에 L2 정규화를 적용하므로 과적합에 강하고,  
학습된 계수가 각 Base 모델의 **최적 앙상블 가중치**를 의미한다.

In [ ]:
# 메타 학습 데이터 구성 (OOF 유효 구간만 사용)
X_meta_train = np.column_stack([
    oof_naive[oof_mask],
    oof_gb[oof_mask],
    oof_lgb[oof_mask]
])
y_meta_train = y_tr_arr[oof_mask]

# Ridge 메타 모델
meta_model = Ridge(alpha=1.0, fit_intercept=True)
meta_model.fit(X_meta_train, y_meta_train)

print('=== OOF 메타 모델 (Ridge) 학습 결과 ===')
print(f'  alpha = 1.0  |  학습 샘플 수 = {len(y_meta_train)}')
print()
print('  학습된 앙상블 가중치:')
for name, coef in zip(['Naive Drift', 'GradientBoosting', 'LightGBM'], meta_model.coef_):
    print(f'    {name:25s}: {coef:+.6f}')
print(f'    {"Intercept":25s}: {meta_model.intercept_:+.6f}')
print()
coef_sum = meta_model.coef_.sum()
print(f'  가중치 합계: {coef_sum:.6f}  (1에 가까울수록 단순 가중 평균에 수렴)')

# 메타 모델의 OOF 내부 적합도
oof_meta_pred = meta_model.predict(X_meta_train)
print()
print('  메타 모델 OOF 내부 성능:')
eval_metrics(y_meta_train, oof_meta_pred, 'Meta Ridge (OOF 내부)')

## 8. 최종 예측 (Validation & Test)

Base 모델을 **전체 학습 데이터**로 재학습한 후 Val/Test 예측을 수행한다.  
이미 위에서 `gb_base`, `lgb_base`를 전체 학습셋으로 학습했으므로 재사용한다.

In [ ]:
def oof_stack_predict(X_arr, naive_preds, gb_model, lgb_model, meta_model):
    """OOF 메타 모델로 최종 예측"""
    gb_preds  = gb_model.predict(X_arr)
    lgb_preds = lgb_model.predict(X_arr)
    X_meta = np.column_stack([naive_preds, gb_preds, lgb_preds])
    return meta_model.predict(X_meta)


# ── Validation 예측 ──────────────────────────────────
oof_val = oof_stack_predict(
    X_val.values, naive_val, gb_base, lgb_base, meta_model
)

# ── Test 예측 (val 마지막 가격 기준 Naive Drift) ─────
oof_test = oof_stack_predict(
    X_test.values, naive_test, gb_base, lgb_base, meta_model
)

print('=== Validation Set 성능 ===')
eval_metrics(y_val, naive_val,  'Naive Drift')
eval_metrics(y_val, hybrid_val, 'Hybrid Baseline (0.8N + 0.2GB)')
eval_metrics(y_val, oof_val,    'OOF Stacking (Meta Ridge)')

print()
print('=== Test Set 성능 ===')
results.append(eval_metrics(y_test, oof_test, 'OOF Stacking (Meta Ridge)'))

## 9. 최종 성능 비교

In [ ]:
results_df = pd.DataFrame(results)
results_df['vs_baseline'] = results_df['rmse'] - BASELINE_RMSE
results_df['개선여부'] = results_df['vs_baseline'].apply(
    lambda x: f'▲ +{abs(x):.2f} 개선' if x < 0 else f'▼ {x:+.2f} 악화'
)
results_df = results_df.set_index('model')

print('=' * 80)
print('              최종 성능 비교표 (Test Set)  /  기준선 RMSE =', BASELINE_RMSE)
print('=' * 80)
print(results_df[['rmse', 'mae', 'mape', 'vs_baseline', '개선여부']].to_string())
print('=' * 80)

best = results_df['rmse'].idxmin()
print(f'\n  ★ 최고 성능 모델: {best}  (RMSE={results_df.loc[best, "rmse"]:.2f})')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# ── Validation 예측 비교 ─────────────────────────────
ax = axes[0]
ax.plot(y_val.index, y_val.values,  'k-o', label='실제값',                  linewidth=2, markersize=5)
ax.plot(y_val.index, naive_val,     'b--', label='Naive Drift',             alpha=0.7)
ax.plot(y_val.index, hybrid_val,    'g--', label='Hybrid (0.8N+0.2GB)',     alpha=0.7)
ax.plot(y_val.index, oof_val,       'r-o', label='OOF Stacking',            linewidth=2, markersize=5)
ax.set_title('Validation Set 예측 비교', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylabel('Nickel Price ($/ton)')
ax.grid(alpha=0.3)

# ── Test 예측 비교 ───────────────────────────────────
ax = axes[1]
ax.plot(y_test.index, y_test.values, 'k-o', label='실제값',                  linewidth=2, markersize=5)
ax.plot(y_test.index, naive_test,    'b--', label='Naive Drift',             alpha=0.7)
ax.plot(y_test.index, hybrid_test,   'g--', label='Hybrid (0.8N+0.2GB)',     alpha=0.7)
ax.plot(y_test.index, oof_test,      'r-o', label='OOF Stacking',            linewidth=2, markersize=5)
ax.set_title('Test Set 예측 비교', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylabel('Nickel Price ($/ton)')
ax.set_xlabel('날짜')
ax.grid(alpha=0.3)

plt.suptitle('OOF Stacking vs 베이스라인 예측 비교\n(니켈 현물가격, 주간)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. 결론 및 분석

### OOF 스태킹 방법론 평가

| 항목 | 내용 |
|------|------|
| **방법론** | TimeSeriesSplit(K=5) → OOF 예측 수집 → Ridge 메타 모델 |
| **Base Models** | Naive Drift, GradientBoosting, LightGBM |
| **Meta Model** | Ridge(alpha=1.0) |
| **데이터 누출 방지** | TimeSeriesSplit (과거→미래 방향 고정) + 1주 shift |

### Residual 방식 vs OOF 방식 최종 비교

| 구분 | Residual 방식 | OOF 방식 |
|------|-------------|----------|
| **장점** | 구현 단순, 잔차 직접 보정 | 과적합 감소, 가중치 자동 최적화 |
| **단점** | In-sample 잔차 사용 → 낙관적 추정 | 계산 비용 K배 증가 |
| **적합 상황** | 단일 강력 Base 모델 존재 시 | 다수 Base 모델 앙상블 시 |

### 추후 개선 방향
1. **메타 피처 확장**: OOF 예측뿐 아니라 원본 피처 일부도 메타 입력으로 추가
2. **Base 모델 다양화**: ARIMA, LSTM 등 이질적 모델 추가로 앙상블 다양성 증가
3. **메타 모델 고도화**: Ridge → LightGBM 메타 모델 (비선형 관계 포착)
4. **Stacking 층 증가**: 2-Level → 3-Level Stacking